Author: Michel Lutz
- Date: 13Jan2026
- Version History:
    - 13Jan2026: update to handle multiple forms with the same lable correctly

## Imports

In [12]:
import pymupdf
import numpy as np
import pandas as pd

## Load files

In [13]:
# set paths
sds_path = "sds_cdisc.xlsx"
unique_crf_path ="acrf_raw.pdf" # source of bookmarks


new_crf_path="acrf_bookmarks.pdf"

In [14]:
# import starting toc
doc_unique = pymupdf.open(unique_crf_path)
#doc_target = pymupdf.open(new_crf_path)

toc_input = doc_unique.get_toc(simple=True)

In [15]:
# import sds file
sds = pd.read_excel(sds_path, sheet_name="Schedule - Grid", header=1, engine="openpyxl")
sds.columns = ["Form Label", "Form Name", *sds.columns[2:].to_list()]
#sds = sds.drop(["not used"], axis=1)

# get all visits in the trail
visit_labels = sds.columns.tolist()[2:]

## Extract booksmarks
create bookmark dictionary for the form
label:[page,name]

In [16]:
# extract bookmarks from starting crf
level_1= [(i,b) for (i,b) in enumerate(toc_input) if b[0]==1]

form_start_numbers = [b[2] for b in toc_input]

form_start_labels = [" (".join(b[1].split(" (")[:-1]) for b in toc_input]
#form_start_labels = [b[1] for b in toc_start]
form_start_names = [b[1].split(" (")[-1].split(")")[0] for b in toc_input]

input_bookmarks_dict = {}
for name, page, label in zip(form_start_names, form_start_numbers, form_start_labels):
    # ML 20-Mar-26: correction for appendix
    if "Appendix" in name:
        label = name
    if label in input_bookmarks_dict.keys():
        input_bookmarks_dict[label].append((page, name))
    else:
        input_bookmarks_dict[label]=[(page, name)]

In [ ]:
#input_bookmarks_dict

{'Adverse Events': [(22, 'ae_00_v001')],
 'Auditory Verbal Learning Test (AVLT-REY)': [(17, 'avlt_rey_v001')],
 'Concomitant Medications': [(25, 'cm__01_v001')],
 'Death Details': [(26, 'death_00_v002')],
 'Demographics': [(5, 'dm_ic_04_v001')],
 'Electroencephalogram': [(20, 'ec_01_v002')],
 'Eligibility': [(6, 'elig_00_v001')],
 'End Of Treatment': [(27, 'eot_00_v001')],
 'Exposure': [(14, 'exp_000')],
 'Hamilton Depression Rating Scale - 17 (HAMD-17)': [(19, 'hamd_17_00_v001')],
 'Informed Consent': [(5, 'ic_01_v001')],
 'Injection Site Reactions': [(23, 'inj_site_00_v001')],
 'Medical History': [(7, 'mh_00_v001')],
 'Ophthalmic Examination': [(21, 'opth_exm_v001')],
 'Patient Health Questionnaire-9 (PHQ-9)': [(15, 'phq_9_v001')],
 'Patient Summary': [(28, 'pat_sum_000')],
 'Satisfaction With Life Survey (SWLS)': [(16, 'swls_000')],
 'Vital Signs (Temperature)': [(11, 'vs_temp_000')],
 'Vital Signs (Weight And Temperature)': [(12, 'vs_wgh_temp_000')],
 'Vital Signs (Weight, Height A

## Add bookmarks by form

In [19]:
def get_form_visits(sds : pd.DataFrame, form_label: str, visit_labels: list[str]):
    # extract the visit/forms for a form
    sds_entry= sds[sds["Form Label"] == form_label]
    if not sds_entry.empty:
        visit_and_names = []
        for index, row in sds_entry.iterrows():
            form_name = row["Form Name"]
            mask = row.notna().to_numpy()[2:]
            form_visits = np.array(visit_labels)[mask].tolist()
            visit_and_names.append((form_name, form_visits))
        return visit_and_names
    else:
        print("No entry in sds file:", form_label)

In [20]:
# iterate over the form lables alphabeticaly
form_labels = list(input_bookmarks_dict.keys())
form_labels.sort()

new_toc_by_form=[]
# add first level bookmark
new_toc_by_form.append([1, 'By Form', 1])

for form_label in form_labels:
    org_bm = input_bookmarks_dict[form_label]
    
    # form_name:page
    form_name_page_dict = {name:page for page,name in org_bm}
    
    # get all visits for the form
    visit_and_names = get_form_visits(sds, form_label, visit_labels)

    # first page for form_name
    first_page = org_bm[0][0]

    # 2-level bookmark - set to first form with the form_label
    new_toc_by_form.append([2, form_label, first_page])

    if visit_and_names:
        # 3-level bookmark
        for form_name, visit_list in visit_and_names:
            for v in visit_list:
                new_toc_by_form.append([3, v, form_name_page_dict[form_name]])
    else:

        new_toc_by_form.append([3, "Appendix", form_name_page_dict[form_label]])
        print(f"CAVE for {form_label} no visits found")

## Add bookmarks by visit

In [21]:
new_toc_by_visit=[]

# add 1-level bookmark
new_toc_by_visit.append([1, 'By Visit', 1])

for visit in visit_labels:
    
    # get sds informaiton by visit - sort alphabeticaly
    sds_visit = sds[sds[visit].notna()][["Form Label", "Form Name"]]
    sds_visit = sds_visit.sort_values(by="Form Label", ascending=True)

    # get first entry by visit
    first_label = sds_visit.iloc[0]["Form Label"]
    first_name=sds_visit.iloc[0]["Form Name"]
    org_bm = input_bookmarks_dict[first_label]
    form_name_page_dict = {name:page for page,name in org_bm} # form_name:page
    fist_page= form_name_page_dict[first_name]

    # add 2-level bookmark
    new_toc_by_visit.append([2, visit, fist_page])

    # add 3-level bookmarks 
    for i, row in sds_visit.iterrows():
        org_bm = input_bookmarks_dict[row["Form Label"]]
        form_name_page_dict = {name:page for page,name in org_bm} # form_name:page
        page = form_name_page_dict[row["Form Name"]]
        new_toc_by_visit.append([3, row["Form Label"], page])


## Finalisation

In [22]:
# sanity
print(len(new_toc_by_form))
print(len(new_toc_by_visit))

# build together
new_toc = new_toc_by_form + new_toc_by_visit
print(len(new_toc))

80
74
154


### Optional - Swap elements in orginal toc
necessary if e.g. plan times should be organised by time but they are sortet alphabetically in toc

In [23]:
def swap_elements(lst, i, j):
    lst[i], lst[j] = lst[j], lst[i]
    return lst

def change_pos(lst, old, new):
    item = lst.pop(old)     # remove element
    lst.insert(new, item)  # insert it at index 3
    return lst

# correction for study 1404.103
correction = False
if correction:
    new_toc = swap_elements(new_toc, 21, 23)
    new_toc = swap_elements(new_toc, 22, 24)
    new_toc = swap_elements(new_toc, 65, 67)
    new_toc = swap_elements(new_toc, 66, 68)
    new_toc = swap_elements(new_toc, 140, 141)
    new_toc = swap_elements(new_toc, 152, 153)
    
    new_toc = change_pos(new_toc, 166, 153)
    new_toc = change_pos(new_toc, 148, 141)
    new_toc = change_pos(new_toc, 93, 67)
    new_toc = change_pos(new_toc, 94, 68)
    new_toc = change_pos(new_toc, 37, 23)
    new_toc = change_pos(new_toc, 38, 24)
    for i, t in enumerate(new_toc): print(i,t)

### Set and save

In [24]:
# write new toc to file
doc_unique.set_toc(new_toc)

# save new CRF
doc_unique.save(new_crf_path)
